# TrustRegionRadius.jl — tutorial

Companion notebook to *A survey of trust-region radius update mechanisms*, Parts I–III.

This walks through the package from a first solve to a full benchmark campaign. It is meant
to be run top to bottom; §1–§3 need only the package, §4 onward adds `Plots`, §9 runs the
test suite and §10 the benchmark harness.

**Contents**

1. Setup and a first solve
2. The three axes
3. Thresholds and scaling factors
4. Tracing, and the observables that matter
5. Diagnosing the Cauchy-point degeneration
6. A model that converges to a saddle
7. Comparing mechanisms with performance profiles
8. Writing a new rule
9. Running the test suite
10. Running the benchmark campaign

## Requirements

```julia
using Pkg
Pkg.develop(path = "..")          # or Pkg.add(url = "https://github.com/USER/TrustRegionRadius.jl")
Pkg.add(["ADNLPModels", "NLPModels", "Plots", "Printf", "LinearAlgebra"])
```

!!! note "Interface change"
    Acceptance of the trial step is now governed by its own threshold `η`, separate from the
    scaling thresholds `η₁` and `η₂`, and every rule carries factors satisfying
    `0 < γ₁ ≤ γ₂ < 1 < γ₃`. `update_radius!` takes an `accepted::Bool` as its third
    argument. Code written against the previous interface will fail loudly rather than
    silently — see `MIGRATION.md` and §3, §8 below.

## 1. Setup and a first solve

`tr_solve` takes any `AbstractNLPModel`. Here an `ADNLPModel`, which differentiates the
objective automatically.

In [1]:
using TrustRegionRadius
using ADNLPModels, NLPModels
using LinearAlgebra, Printf
using LinearOperators, SolverCore
using Test

In [2]:
rosenbrock(x) = (1 - x[1])^2 + 100 * (x[2] - x[1]^2)^2

nlp = ADNLPModel(rosenbrock, [-1.2, 1.0], name = "ROSENBR")

stats = tr_solve(nlp; rule = RDelta())

@printf("status     : %s\n", stats.status)
@printf("solution   : (%.8f, %.8f)\n", stats.solution[1], stats.solution[2])
@printf("objective  : %.3e\n", stats.objective)
@printf("‖g‖        : %.3e\n", stats.dual_feas)
@printf("iterations : %d\n", stats.iter)
@printf("f evals    : %d,   g evals: %d\n", neval_obj(nlp), neval_grad(nlp))

status     : first_order
solution   : (1.00000000, 1.00000000)
objective  : 4.712e-22
‖g‖        : 9.645e-10
iterations : 32
f evals    : 33,   g evals: 27


`stats` is a JSO `GenericExecutionStats` (aliased `TRResult`), so anything in the
JuliaSmoothOptimizers ecosystem accepts it unchanged. Evaluation counts live on the *model*,
not on the result — that is why `neval_obj(nlp)` rather than `stats.neval_obj`.

Only `:first_order` counts as solved. A run that stops at `:max_iter` with a large gradient
has not solved the problem, whatever its iteration count.

## 2. The three axes

The point of the package: radius rule, model Hessian and subproblem solver vary
independently, and every combination goes through the same driver. A comparison therefore
measures the axis you changed rather than the difference between two code bases.

In [4]:
rules = [("RDelta",           RDelta()),
         ("RStep",            RStep()),
         ("RDFO(ζ=1)",        RDFO(ζ = 1.0)),
         ("RGrad",            RGrad()),
         ("RGradCapped",       RGradCapped(μ_max = 1.0)),
         ("RAdaptiveStep",    RAdaptiveStep()),
         ("RAdaptiveGrad",    RAdaptiveGrad()),
         ("RAdaptiveFanYuan", RAdaptiveFanYuan()),
         ("RRTR",             RRTR()),
         ("RRTRGrad",         RRTRGrad())]

@printf("%-18s %-12s %8s %10s\n", "rule", "status", "iters", "‖g‖")
println("-"^52)
for (name, rule) in rules
    nlp = ADNLPModel(rosenbrock, [-1.2, 1.0])
    st  = tr_solve(nlp; rule = rule, params = TRParams(tol = 1e-6, max_iterations = 5_000))
    @printf("%-18s %-12s %8d %10.2e\n", name, st.status, st.iter, st.dual_feas)
end

rule               status          iters        ‖g‖
----------------------------------------------------
RDelta             first_order        32   9.65e-10
RStep              first_order        42   1.23e-07
RDFO(ζ=1)          first_order        37   9.65e-10
RGrad              first_order        36   5.62e-08
RGradCapped        first_order        54   5.51e-13
RAdaptiveStep      first_order        44   2.13e-08
RAdaptiveGrad      first_order        42   6.50e-10
RAdaptiveFanYuan   first_order        41   6.08e-10
RRTR               first_order        38   3.17e-09
RRTRGrad           first_order        35   7.77e-07


In [5]:
# Same rule, different model Hessians.
models = [("exact",       ExactHessian()),
          ("L-BFGS(5)",   LBFGSModel(mem = 5)),
          ("SR1(5)",      SR1Model(mem = 5)),
          ("I  (no curvature)", ScaledIdentity(c = 1.0))]

@printf("%-20s %-12s %8s %10s\n", "model", "status", "iters", "‖g‖")
println("-"^54)
for (name, model) in models
    nlp = ADNLPModel(rosenbrock, [-1.2, 1.0])
    st  = tr_solve(nlp; rule = RDelta(), model = model,
                   params = TRParams(tol = 1e-6, max_iterations = 20_000))
    @printf("%-20s %-12s %8d %10.2e\n", name, st.status, st.iter, st.dual_feas)
end

model                status          iters        ‖g‖
------------------------------------------------------
exact                first_order        32   9.65e-10
L-BFGS(5)            first_order        71   8.55e-08
SR1(5)               first_order       101   4.63e-10
I  (no curvature)    first_order      8400   1.00e-06


`ScaledIdentity` carries no curvature at all, so with it the step is `-g/c` truncated
to the region and the method is exactly gradient descent. It is a diagnostic instrument: any
difference *within* that column is attributable to the radius rule alone.

!!! note
    Keep `TRParams` identical across a comparison. Tuning γ's per rule measures tuning effort
    rather than algorithmic merit, and makes a performance profile uninterpretable. The one
    threshold worth varying deliberately is the acceptance threshold `η`, because it is not a
    per-rule tuning knob: it changes the algorithm for every mechanism at once. §3 shows how.

## 3. Thresholds and scaling factors

Two conventions run through the whole package, and both are checked rather than documented.

### Acceptance is separate from scaling

`TRParams` carries three thresholds with `0 ≤ η ≤ η₁ ≤ η₂ < 1`:

| threshold | role |
|:---|:---|
| `η`  | **acceptance**: the step is taken iff `ρ_k ≥ η`. This is what defines 𝒮 and 𝒰. |
| `η₁` | **scaling**: below it the radius contracts. |
| `η₂` | **scaling**: above it the iteration is very successful and the radius may grow. |

`η` defaults to `η₁`, which is the classical coupled algorithm, so nothing changes unless you
ask for it. Setting `η < η₁` opens a regime that does not exist in the coupled formulation:
iterations with `ρ_k ∈ [η, η₁)` **accept the step and contract the radius at the same time**.
Setting `η = 0` accepts every step with a positive predicted reduction — the case Part I
covers and the framework of Curtis & Scheinberg (2020) excludes.

No rule ever sees `η`. The solver applies it and reports the outcome, which is why
`update_radius!` receives an explicit `accepted::Bool`: a rule cannot derive acceptance from
the ratio it is given, and the retrospective rules are not even given `ρ_k`.

In [6]:
"Summarise one run under a given threshold configuration."
function threshold_report(rule, p; f = rosenbrock, x0 = [-1.2, 1.0])
    nlp = ADNLPModel(f, x0)
    st  = tr_solve(nlp; rule = rule, params = p, trace = true)
    ss  = st.solver_specific
    ρ, acc = ss[:ratio_trajectory], ss[:accepted_trajectory]

    # Accepted while the *update* still called for contraction. Note the test is
    # ρ < η₁, not Δ shrinking: RDelta contracts by γ₂ on every mildly successful
    # iteration too, so a shrinking radius does not isolate the new regime.
    # This band is empty by construction whenever η = η₁.
    band = count(i -> acc[i] && ρ[i] < p.η₁, eachindex(acc))
    return (status = st.status, iters = st.iter,
            accepted = count(acc), band = band)
end

settings = [("coupled    η = η₁ = 0.1", TRParams(η₁ = 0.1, η₂ = 0.9, tol = 1e-6)),
            ("decoupled  η = 0.01",     TRParams(η = 0.01, η₁ = 0.1, η₂ = 0.9, tol = 1e-6)),
            ("decoupled  η = 0",        TRParams(η = 0.0,  η₁ = 0.1, η₂ = 0.9, tol = 1e-6))]

@printf("%-24s %-12s %8s %10s %24s\n",
        "thresholds", "status", "iters", "accepted", "accepted, ρ < η₁")
println("-"^82)
for (label, p) in settings
    r = threshold_report(RDelta(), p)
    @printf("%-24s %-12s %8d %10d %24d\n",
            label, r.status, r.iters, r.accepted, r.band)
end

thresholds               status          iters   accepted         accepted, ρ < η₁
----------------------------------------------------------------------------------
coupled    η = η₁ = 0.1  first_order        32         26                        0
decoupled  η = 0.01      first_order        29         24                        1
decoupled  η = 0         first_order        29         24                        1


The last column is the decoupling made visible: with `η = η₁` it is zero by construction,
and it grows as `η` falls. Those iterations move the iterate on a step the classical
algorithm would have thrown away, while still shrinking the region because the model was a
poor predictor. Whether that is a good idea is an empirical question — which is the point of
being able to set the two independently.

### One convention for the scaling factors

Every rule carries `γ₁, γ₂, γ₃` with

$$0 < \gamma_1 \le \gamma_2 < 1 < \gamma_3,$$

`γ₁` the aggressive contraction, `γ₂` the mild one, `γ₃` the expansion. `check_factors`
enforces this in every constructor, so a factor in the wrong slot is an error at construction
rather than a silently different algorithm.

This is a change: `RGrad`, `RGradCapped`, `RRTR` and `RRTRGrad` previously used `γ₂` for
expansion. `RGrad(γ₂ = 2.0)` now throws.

In [ ]:
attempts = [
    ("RGrad(γ₂ = 2.0)                — expansion in the γ₂ slot", () -> RGrad(γ₂ = 2.0)),
    ("RDelta(γ₁ = 0.8, γ₂ = 0.3)     — γ₁ > γ₂",                  () -> RDelta(γ₁ = 0.8, γ₂ = 0.3)),
    ("RDelta(γ₃ = 0.9)               — γ₃ ≤ 1",                   () -> RDelta(γ₃ = 0.9)),
    ("RAdaptiveStep(γ₂=0.5, γ₃=1.4)  — γ₃ ≤ 1 + γ₂",              () -> RAdaptiveStep(γ₂ = 0.5, γ₃ = 1.4)),
    ("RGrad(γ₁=0.25, γ₂=0.5, γ₃=2.0) — correct",                  () -> RGrad(γ₁ = 0.25, γ₂ = 0.5, γ₃ = 2.0)),
]

for (desc, thunk) in attempts
    try
        thunk()
        println("accepted : ", desc)
    catch err
        msg = first(split(sprint(showerror, err), '\n'))
        println("rejected : ", desc, "\n           ", msg)
    end
end

In [ ]:
# Where each mechanism sits in the classification of Part II. `asymptotic_regime` is a
# three-way split, which `is_criticality_anchored` alone cannot express: the step-anchored
# rules drive Δ_k → 0 in the local regime without the radius being tied to a criticality
# measure at all.
allrules = [RDelta(), RStep(), RDFO(), RGrad(), RGradCapped(μ_max = 8.0),
            RAdaptiveStep(), RAdaptiveGrad(), RAdaptiveFanYuan(), RRTR(), RRTRGrad()]

@printf("%-20s %-16s %12s %10s\n", "rule", "regime", "anchored", "uses ρ̃")
println("-"^62)
for r in allrules
    @printf("%-20s %-16s %12s %10s\n",
            nameof(typeof(r)), asymptotic_regime(r),
            is_criticality_anchored(r), needs_retrospective(r))
end

## 4. Tracing, and the observables that matter

`trace = true` records per-iteration trajectories. Two are worth reading closely:
`:active_trajectory`, whether the trust-region constraint was binding, and
`:accepted_trajectory`, which iterations belonged to 𝒮. The second cannot be reconstructed
from the ratio trace once `η < η₁`, since the rule's thresholds no longer coincide with the
one that decided the step.

In [ ]:
nlp = ADNLPModel(rosenbrock, [-1.2, 1.0])
st  = tr_solve(nlp; rule = RGrad(), trace = true,
               params = TRParams(tol = 1e-8, max_iterations = 5_000))

ss = st.solver_specific
for k in sort(collect(keys(ss)))
    v = ss[k]
    @printf("%-24s %s\n", k, v isa AbstractVector ? "length $(length(v))" : string(v))
end
println("\niterations reported by stats: ", st.iter)

In [ ]:
Δ, g = ss[:delta_trajectory], ss[:grad_trajectory]
a, acc = ss[:active_trajectory], ss[:accepted_trajectory]

# Δ, ‖g‖ and f have a value before the first iteration, so they are one entry longer
# than the per-iteration records. Align on the tail when plotting them together.
@printf("iterations            : %d\n", length(a))
@printf("Δ₀ → Δ_end            : %.3e → %.3e\n", Δ[1], Δ[end])
@printf("‖g₀‖ → ‖g_end‖        : %.3e → %.3e\n", g[1], g[end])
@printf("accepted iterations   : %d / %d\n", count(acc), length(acc))
@printf("active iterations     : %d / %d\n", count(a), length(a))

last = 10
@printf("\nactive in last %d      : %.2f\n", last,
        count(a[max(1, end - last + 1):end]) / min(last, length(a)))
@printf("rejections contracted : %s\n",
        all(i -> acc[i] || Δ[i + 1] < Δ[i], eachindex(acc)))

Why this matters: two mechanisms can have identical iteration counts and identical
first-order behaviour — both drive `‖g‖ → 0` — while one keeps the constraint permanently
active and the other does not. Nothing else in a standard diagnostic distinguishes them, and
the difference decides whether the method reaches its superlinear regime.

The comparison below puts a deliberately-too-small `μ_max` beside the uncapped rule.

In [ ]:
configs = [("RDelta",             RDelta()),
           ("RGrad (uncapped)",   RGrad(μ = 1.0)),
           ("RGradCapped μ̄=1",    RGradCapped(μ = 1.0,  μ_max = 1.0)),
           ("RGradCapped μ̄=0.05", RGradCapped(μ = 0.05, μ_max = 0.05))]

@printf("%-22s %-12s %8s %14s\n", "configuration", "status", "iters", "tail active")
println("-"^60)
for (name, rule) in configs
    nlp = ADNLPModel(rosenbrock, [-1.2, 1.0])
    st  = tr_solve(nlp; rule = rule, trace = true,
                   params = TRParams(tol = 1e-6, max_iterations = 20_000))
    a   = st.solver_specific[:active_trajectory]
    k0  = max(1, floor(Int, 0.9 * length(a)))
    frac = isempty(a) ? NaN : count(a[k0:end]) / length(a[k0:end])
    @printf("%-22s %-12s %8d %14.3f\n", name, st.status, st.iter, frac)
end

A tail-active fraction near 1 means the constraint never stops binding, so the method
converges at best linearly. The theory says this happens whenever the cap sits below
`κ̄ = 4/λ*_min(∇²f(x*))` — a constant that depends on the *solution*, and so cannot be chosen
in advance. Uncapped `RGrad` escapes by construction: `μ` grows geometrically past any
threshold.

### The inactivity countdown

The tail-active fraction answers "is the constraint still binding near the end?" but not
"when did it stop?". For that, count the active iterations that remain *ahead* of each index:

$$R_k = \#\{\, j > k \;:\; \|s_j\| = \Delta_j \,\}, \qquad k = 0, \dots, K-1.$$

`R` starts at the total number of active iterations, steps down by one at each active
iteration and is flat on inactive ones. It reaches zero exactly when every iteration that
follows is inactive, so the first `k` with `R_k = 0` is the onset `k*` of the inactive regime
— the index the inactivity theorems of Part II are about. A curve that never reaches zero
means the constraint was still binding on the last iteration, and the run never entered the
regime in which the local rate is available.

The range stops at `K-1` on purpose: `R_K = 0` for every run, because no iteration follows
the last one, so including it would make every curve touch zero and erase the distinction.

In [ ]:
using Plots     # first cell that needs it

"R_k = number of active iterations after k, for k = 0, …, K-1."
function remaining_active(active::AbstractVector{Bool})
    K = length(active)
    R = zeros(Int, K + 1)
    for k in K:-1:1
        R[k] = R[k + 1] + (active[k] ? 1 : 0)
    end
    return R[1:K]
end

"Index of the last active iteration, and the length of the inactive run after it."
function inactivity_onset(active::AbstractVector{Bool})
    k = length(active)
    while k >= 1 && !active[k]
        k -= 1
    end
    return k, length(active) - k
end

countdown_cfgs = [("RDelta",             RDelta()),
                  ("RGrad (uncapped)",   RGrad(μ = 1.0)),
                  ("RGradCapped μ̄=1",    RGradCapped(μ = 1.0,  μ_max = 1.0)),
                  ("RGradCapped μ̄=0.05", RGradCapped(μ = 0.05, μ_max = 0.05)),
                  ("RDFO ζ=0.01",        RDFO(ζ = 0.01))]

plt = plot(; xlabel = "iteration k", ylabel = "remaining active iterations",
             title = "ROSENBR: active iterations remaining after k",
             legend = :topright, lw = 2)

@printf("%-22s %-12s %8s %8s %8s %8s\n",
        "configuration", "status", "iters", "active", "k*", "tail")
println("-"^70)
for (name, rule) in countdown_cfgs
    nlp = ADNLPModel(rosenbrock, [-1.2, 1.0])
    st  = tr_solve(nlp; rule = rule, trace = true,
                   params = TRParams(tol = 1e-6, max_iterations = 20_000))
    a = st.solver_specific[:active_trajectory]
    isempty(a) && continue

    R = remaining_active(a)
    k_star, tail = inactivity_onset(a)
    label = name * (tail == 0 ? "  [never]" : "  [k*=$(k_star)]")
    plot!(plt, 0:(length(R) - 1), R; label = label, seriestype = :steppost)
    tail == 0 || scatter!(plt, [k_star], [0.0]; label = "", ms = 5, mc = :black)

    @printf("%-22s %-12s %8d %8d %8s %8d\n", name, st.status, st.iter, count(a),
            tail == 0 ? "never" : string(k_star), tail)
end
plt

The curves that fall to zero are the mechanisms that reach the asymptotic regime; the flat
ones never do. Every configuration in the table still stops at `:first_order` with `‖g‖`
below the tolerance, which is the point: the difference is invisible to the convergence test
and visible here.

Note that a positive `tail` is evidence of inactivity *over the iterations observed*, not a
proof of eventual inactivity, and a `tail` of zero may only mean the budget ran out before
the onset. Comparing `tail` against `iters` distinguishes the two readings.
`benchmark/experiments/exp8_single_problem.jl` produces this figure for any problem, together
with the Δ, `‖g‖` and ρ trajectories on a shared axis.

## 5. Diagnosing the Cauchy-point degeneration

A small radius makes truncated CG stop on its first iteration. That first CG direction is
`-g`, so the returned step is exactly the Cauchy point and the model Hessian has had no
influence on the *direction* at all — the method is gradient descent in disguise.

`cg_step_info` measures it directly.

In [ ]:
nlp = ADNLPModel(rosenbrock, [-1.2, 1.0])
x   = copy(nlp.meta.x0)
g   = grad(nlp, x)
gn  = norm(g)

@printf("%10s %12s %10s %10s %20s\n", "μ_max", "Δ = μ‖g‖", "CG iters", "active", "cos(s, -g)")
println("-"^68)
for μ in [0.001, 0.01, 0.1, 0.3, 1.0, 10.0]
    info = cg_step_info(SteihaugCG(), ExactHessian(), nlp, x, g, μ * gn)
    @printf("%10.3f %12.4e %10d %10s %20.15f\n",
            μ, μ * gn, info.cg_iters, string(info.active), info.cos_cauchy)
end

Where `CG iters == 1`, `active == true` and `cos(s, -g) == 1` to machine precision,
the model Hessian played no part in choosing the direction. Nothing in ρ, `‖g‖` or the radius
trace reveals this; only the CG iteration count and the cosine do.

This is the mechanism behind the `μ_max` threshold seen in §4, and the reason a
*conservative* cap is the dangerous choice rather than the safe one.

## 6. A model that converges to a saddle

`SPDTarget` builds a positive definite model whose unconstrained minimiser is pinned to a
chosen point. Aim it at a saddle and every hypothesis of the first-order theory holds — ρ
successful throughout, `‖g‖ → 0`, radius well behaved — yet the limit is a saddle. The
failure is the model, and no radius rule can repair it.

The test function has four critical points:

In [ ]:
# f(x,y) = x⁴ − x³ + (¼ − x/2)y² + ¼y⁴
quartic(p) = p[1]^4 - p[1]^3 + (0.25 - p[1]/2) * p[2]^2 + p[2]^4 / 4

crit = [("origin (degenerate)", [0.0, 0.0]),
        ("saddle",              [0.75, 0.0]),
        ("min⁺",  [(1 + sqrt(5))/4,  sqrt((sqrt(5) - 1)/4)]),
        ("min⁻",  [(1 + sqrt(5))/4, -sqrt((sqrt(5) - 1)/4)])]

probe = ADNLPModel(quartic, [0.1, 0.1])
@printf("%-22s %-24s %12s %22s\n", "point", "x", "f", "eig(∇²f)")
println("-"^84)
for (name, p) in crit
    H = Symmetric(Matrix(hess(probe, p)))
    w = eigvals(H)
    @printf("%-22s (%+.6f, %+.6f) %12.8f   (%+.5f, %+.5f)\n",
            name, p[1], p[2], quartic(p), w[1], w[2])
end

In [ ]:
target = [0.75, 0.0]          # the saddle
x0     = [-0.5, 0.6]

nlp   = ADNLPModel(quartic, x0)
model = SPDTarget(target = target)

# The construction exists only where the target lies downhill: φ(x) = gᵀ(target − x) < 0.
@printf("φ(x₀) = %.6f   (must be < 0)\n\n", phi_target(model, nlp, x0))

st = tr_solve(nlp; rule = RDelta(), model = model, trace = true,
              params = TRParams(tol = 1e-10))

@printf("status           : %s\n", st.status)
@printf("limit            : (%.10f, %.3e)\n", st.solution[1], st.solution[2])
@printf("distance to saddle: %.3e\n", norm(st.solution .- target))
@printf("iterations       : %d\n", st.iter)

ρ = st.solver_specific[:ratio_trajectory]
@printf("\nevery ρ successful? %s   (min ρ = %.4f)\n", all(ρ .>= 0.9), minimum(ρ))
@printf("‖g‖ decreasing?     %s\n",
        issorted(st.solver_specific[:grad_trajectory], rev = true))

Every diagnostic a practitioner monitors is healthy, and the limit is a saddle.

Ask for a target that lies *uphill* and the construction correctly refuses, because no
positive definite model with that minimiser exists there:

In [ ]:
bad = SPDTarget(target = [5.0, 5.0])
@printf("φ = %+.5f  (≥ 0 ⇒ no such model)\n", phi_target(bad, nlp, x0))
try
    dense_hessian(bad, nlp, x0)
catch err
    println("correctly refused: ", typeof(err))
end

## 7. Comparing mechanisms with performance profiles

`performance_profile` implements Dolan & Moré (2002) and is a pure function of a cost matrix,
so it works on numbers from anywhere.

In [ ]:
using Plots

# A small analytic test set.
testset = [
    ("ROSENBR",  x -> 100(x[2]-x[1]^2)^2 + (1-x[1])^2,                  [-1.2, 1.0]),
    ("ROSENBR2", x -> 100(x[2]-x[1]^2)^2 + (1-x[1])^2,                  [ 2.0, 2.0]),
    ("BEALE",    x -> (1.5-x[1]+x[1]*x[2])^2 + (2.25-x[1]+x[1]*x[2]^2)^2 +
                      (2.625-x[1]+x[1]*x[2]^3)^2,                       [ 1.0, 1.0]),
    ("HIMMELBLAU", x -> (x[1]^2+x[2]-11)^2 + (x[1]+x[2]^2-7)^2,         [ 0.0, 0.0]),
    ("POWELLSG", x -> (x[1]+10x[2])^2 + 5(x[3]-x[4])^2 +
                      (x[2]-2x[3])^4 + 10(x[1]-x[4])^4,           [3.0,-1.0,0.0,1.0]),
    ("WOOD",     x -> 100(x[2]-x[1]^2)^2 + (1-x[1])^2 + 90(x[4]-x[3]^2)^2 +
                      (1-x[3])^2 + 10.1*((x[2]-1)^2+(x[4]-1)^2) +
                      19.8*(x[2]-1)*(x[4]-1),                    [-3.0,-1.0,-3.0,-1.0]),
    ("ILLCOND",  x -> x[1]^2 + 1000x[2]^2 + 0.01x[1]*x[2],              [ 1.0, 1.0]),
    ("TRIGQUAD", x -> cos(x[1])*cos(x[2]) + 0.1(x[1]^2+x[2]^2),         [ 1.0, 0.5]),
]

problems = [() -> ADNLPModel(f, x0, name = nm) for (nm, f, x0) in testset]

configs = [TRConfig("RDelta";    rule = RDelta()),
           TRConfig("RStep";     rule = RStep()),
           TRConfig("RDFO";      rule = RDFO(ζ = 1.0)),
           TRConfig("RGrad";     rule = RGrad()),
           TRConfig("RGrad-cap"; rule = RGradCapped(μ = 1.0, μ_max = 1.0))]

T, S = run_matrix(problems, configs; cost = :iter)
println()
summarise(T, [c.label for c in configs])

In [ ]:
labels  = [c.label for c in configs]
τ, prof = performance_profile(T)

plot(τ, prof;
     xscale = :log10, label = reshape(labels, 1, :), lw = 2,
     xlabel = "τ  (performance ratio, iterations)",
     ylabel = "π(τ)", legend = :bottomright, ylims = (0, 1.02),
     title  = "Performance profile")

Read the two ends separately:

* **`π(1)`** — the intercept — is *efficiency*: the fraction of problems on which the rule is
  fastest.
* **`π(∞)`** — the right-hand asymptote — is *reliability*: the fraction it solves at all.

These move independently, and that is the substance of most comparisons here. A parameter
choice can leave efficiency untouched while collapsing reliability.

!!! warning
    Ratios are normalised by the per-problem best, so adding or removing a solver can reorder
    the curves (Gould & Scott 2016). Report pairwise profiles against a fixed baseline
    alongside the full comparison, and use `data_profile` — which is not normalised that way
    — as a cross-check.

!!! warning "Problems with a zero cost"
    A problem already solved at `x₀` returns `:first_order` with `iter == 0`. Such a row
    carries no information about any mechanism, and the two summaries disagree about it:
    `performance_profile` treats a cost of `0` as a failure, so reliability drops for every
    configuration, while `success_table` counts it as solved and pulls the medians down. Some
    CUTEst entries behave this way — the `NE` reformulations in particular can arrive with a
    null objective, so `∇f ≡ 0` everywhere. Screen out problems on which *every*
    configuration reports zero iterations before building the cost matrix, and log the names
    that were dropped.

In [ ]:
# A ζ sweep: the expected signature is flat efficiency, collapsing reliability.
zetas   = [0.01, 0.1, 0.5, 1.0, 10.0, 100.0]
zconfig = sweep_configs("ζ", zetas, ζ -> RDFO(ζ = ζ))

Tz, _ = run_matrix(problems, zconfig; cost = :iter, verbose = false)
zlab  = [c.label for c in zconfig]
summarise(Tz, zlab)

τz, profz = performance_profile(Tz)
p1 = plot(τz, profz; xscale = :log10, label = reshape(zlab, 1, :), lw = 2,
          xlabel = "τ", ylabel = "π(τ)", legend = :bottomright,
          ylims = (0, 1.02), title = "R-DFO: influence of ζ")

rate = [count(isfinite, Tz[:, j]) / size(Tz, 1) for j in 1:length(zconfig)]
eff  = [profz[1, j] for j in 1:length(zconfig)]
p2 = plot(zetas, rate; xscale = :log10, marker = :circle, lw = 2,
          label = "reliability", xlabel = "ζ", ylabel = "fraction",
          ylims = (0, 1.02), legend = :bottomright)
plot!(p2, zetas, eff; marker = :square, lw = 2, ls = :dash, label = "efficiency")

plot(p1, p2; layout = (1, 2), size = (960, 380))

## 8. Writing a new rule

A rule is a subtype of `RadiusRule` plus two methods. `update_radius!` must be extended
under its qualified name, since `using` brings it into scope without making it extensible:

```julia
initial_radius(rule, Δ₀, g_norm)                                            -> Float64
update_radius!(rule, Δ, ρ, accepted, η₁, η₂, s_norm, g_norm_old, g_norm_new) -> Float64
reset_rule!(rule)                                                            -> nothing
```

The arguments in order: the current radius; the ratio driving the update (`ρ_k`, or `ρ̃_k` for
a rule declaring `needs_retrospective`); whether the step was accepted; the two scaling
thresholds; `‖s_k‖`; and `‖g_k‖` before and after the accept/reject decision. Ignored
arguments are written as bare type annotations.

One obligation is not optional. On an unsuccessful iteration the rule **must** return a
radius strictly smaller than the one it was given: that is the contraction condition, and
without it the next iteration re-solves an identical subproblem with an identical model, for
ever. `test/test_thresholds.jl` checks it for every rule in the package, and any new rule
should be added there.

In [ ]:
"""
    RHalfDouble(; γ₁ = 0.25, γ₂ = 0.5, γ₃ = 2.0)

Minimal three-case rule on the radius: contract hard on failure, hold when mildly
successful, expand when very successful. Written to show the interface, not as a
recommendation — it is `RDelta` with the middle branch neutered.
"""
struct RHalfDouble <: RadiusRule
    γ₁::Float64
    γ₂::Float64
    γ₃::Float64
    function RHalfDouble(; γ₁ = 0.25, γ₂ = 0.5, γ₃ = 2.0)
        check_factors(:RHalfDouble; γ₁ = γ₁, γ₂ = γ₂, γ₃ = γ₃)
        new(γ₁, γ₂, γ₃)
    end
end

function TrustRegionRadius.update_radius!(r::RHalfDouble, Δ::Float64, ρ::Float64,
                                          ::Bool, η₁::Float64, η₂::Float64,
                                          ::Float64, ::Float64, ::Float64)
    ρ < η₁  && return r.γ₁ * Δ        # unsuccessful: must be < Δ
    ρ >= η₂ && return r.γ₃ * Δ
    return Δ
end

TrustRegionRadius.asymptotic_regime(::RHalfDouble) = :bounded_below

# The contraction obligation, checked directly.
let r = RHalfDouble(), Δ = 1.0
    for ρ in (-1.0, 0.0, 0.05)
        Δnew = update_radius!(r, Δ, ρ, false, 0.1, 0.9, 0.8, 2.0, 2.0)
        @assert Δnew < Δ "RHalfDouble failed to contract at ρ = $ρ"
    end
    println("contraction on unsuccessful iterations: ok")
end

nlp = ADNLPModel(rosenbrock, [-1.2, 1.0])
st  = tr_solve(nlp; rule = RHalfDouble(), trace = true,
               params = TRParams(tol = 1e-6, max_iterations = 5_000))
@printf("RHalfDouble : %s in %d iterations, ‖g‖ = %.2e, regime = %s\n",
        st.status, st.iter, st.dual_feas, asymptotic_regime(RHalfDouble()))

## 9. Running the test suite

The suite asserts not only that the code runs but that the survey's claims hold: that
uncapped `RGrad` drives μ past any threshold, that `RStep`'s `Δmin` prevents the collapse to
zero, that `cg_step_info` returns `cos(s,−g) = 1` on a tiny radius, that `SPDTarget` throws
exactly when φ ≥ 0, and that every rule contracts on an unsuccessful iteration under
decoupled thresholds. A failure there is a finding, not merely a bug.

In [ ]:
using Pkg
# Runs test/runtests.jl in a clean environment.
Pkg.test("TrustRegionRadius")

In [ ]:
# Or a single file, without the full harness — faster while developing.
using Test
TESTDIR = joinpath(pkgdir(TrustRegionRadius), "test")
include(joinpath(TESTDIR, "test_rules.jl"))
include(joinpath(TESTDIR, "test_thresholds.jl"))   # η/η₁/η₂ and the γ convention

## 10. Running the benchmark campaign

The campaign lives in a separate environment so that `Plots`, `JLD2` and `CUTEst` are not
dependencies of the package itself.

```bash
julia --project=benchmark -e 'using Pkg; Pkg.develop(path="."); Pkg.instantiate()'
julia --project=benchmark benchmark/experiments/run_all.jl
julia --project=benchmark benchmark/experiments/run_all.jl 3 4   # just ζ and μ̄ sweeps

# Experiment 8 takes a problem name: trajectories and the inactivity countdown.
julia --project=benchmark benchmark/experiments/exp8_single_problem.jl WOOD
```

Each run writes a self-documenting archive under `benchmark/results/`:

```
exp_2026-04-16_02-08-34_zeta_sweep/
├── experiment_config.toml     what was run
├── experiment_summary.md      what came out
├── figures/                   PDFs
├── tables/                    text tables and .tex profile coordinates
└── data/                      raw JLD2, one file per (problem, rule)
```

The cell below reproduces the archiving machinery inline, so you can see the shape of an
archive without leaving the notebook.

In [ ]:
# Requires the benchmark environment (TOML, Dates, JLD2, Plots).
BENCH = joinpath(pkgdir(TrustRegionRadius), "benchmark")

include(joinpath(BENCH, "archive.jl"))
include(joinpath(BENCH, "harness.jl"))

arch = ExperimentArchive(joinpath(BENCH, "results"); tag = "notebook_demo")

save_config(arch;
    rules  = [("RDelta", () -> RDelta()), ("RGrad", () -> RGrad())],
    params = TRParams(tol = 1e-6),
    extra  = Dict("experiment" => "notebook_demo"))

records = run_experiment(
    analytic_problems(),
    [("RDelta", () -> (rule = RDelta(),)),
     ("RGrad",  () -> (rule = RGrad(),))];
    params = TRParams(tol = 1e-6, max_iterations = 5_000),
    archive = arch)

save_table(arch, "success_rate.txt",
           success_table(records, analytic_problems(),
                         [("RDelta", nothing), ("RGrad", nothing)]))

finalize_archive(arch; notes = "Generated from the tutorial notebook.")
println("\narchive: ", arch.dir)

In [ ]:
# Inspect what was written.
for (root, _, files) in walkdir(arch.dir)
    rel = relpath(root, arch.dir)
    println(rel == "." ? basename(arch.dir) * "/" : "  " * rel * "/")
    for f in sort(files)
        println("      ", f)
    end
end

println("\n--- experiment_config.toml ---")
print(read(joinpath(arch.dir, "experiment_config.toml"), String))

## Where to go next

* **Docs** — `julia --project=docs docs/make.jl`, then open `docs/build/index.html`.
* **Migration** — `MIGRATION.md` lists the breaking changes: the acceptance threshold `η`,
  the `accepted` argument, the renumbered factors in `RGrad`, `RGradCapped`, `RRTR` and
  `RRTRGrad`, and the renamed Hei parameters.
* **CUTEst** — `cutest_problems(min_var = 2, max_var = 500)` in the harness returns thunks
  ready for `run_experiment`. If CUTEst is not installed the harness falls back to
  `analytic_problems()`, so nothing else breaks.
* **A new rule** — one struct plus `initial_radius` and `update_radius!`; see §8 and
  [Getting started](https://USER.github.io/TrustRegionRadius.jl/dev/quickstart/).

## Four things this notebook demonstrated

1. **The radius rule sets the step length; the model sets the direction.** Across §2 the limit
   tracked the model Hessian, not the rule. No radius rule repairs a model that misreports
   curvature (§6).
2. **A conservative parameter is the dangerous choice.** A small `μ_max` or `ζ` does not merely
   slow the method — it can keep the constraint binding forever (§4) and reduce the step to the
   Cauchy point, discarding the model entirely (§5).
3. **First-order diagnostics cannot see either failure.** In §6 every ρ was successful and
   `‖g‖ → 0`, and the limit was a saddle. The distinguishing measurements are the inactivity
   countdown and the CG iteration count — both cheap, and neither usually recorded.
4. **Acceptance and scaling answer different questions.** Whether to move is not whether to
   trust the model over a longer step, and §3 separates them: with `η < η₁` a step can be
   taken while the region shrinks. The classical algorithm ties the two together by
   convention, not by necessity.